In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense, Flatten
from tensorflow import keras
from keras.optimizers import Adam
from tensorflow.keras.datasets import cifar10

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [ ]:
for i in range(y_train.shape[0]):
  if y_train[i] == 1 or y_train[i] == 9:
    y_train[i] = 1
  else:
    y_train[i] = 0
for i in range(y_test.shape[0]):
  if y_test[i] == 1 or y_test[i] == 9:
    y_test[i] = 1
  else:
    y_test[i] = 0

In [ ]:
x_train = x_train/np.max(x_train)
x_test = x_test/np.max(x_test)

In [ ]:
y_train_cat = keras.utils.to_categorical(y_train, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)

In [ ]:
inputs = keras.Input(shape=(32, 32, 3), name="img")
x = layers.Conv2D(32, 3, activation="relu")(inputs) #Свёрточный слой
x = layers.Conv2D(64, 3, activation="relu")(x)
block_1_output = layers.MaxPooling2D(3)(x) #MaxPooling слой

In [ ]:
x = layers.Conv2D(64, 3, activation="relu", padding="same")(block_1_output)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)

In [ ]:
block_2_output = layers.add([x, block_1_output])

In [ ]:
x = layers.Conv2D(64, 3, activation="relu", padding="same")(block_2_output)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
block_3_output = layers.add([x, block_2_output])

In [ ]:
x = layers.Conv2D(64, 3, activation="relu")(block_3_output)
x = layers.GlobalAveragePooling2D()(x) #GlobalAveragePooling2D слой
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x) #Dropout() для увеличения обобщающей способности сети.

In [ ]:

outputs = layers.Dense(2, activation='softmax')(x)

model = keras.Model(inputs = inputs, outputs = outputs)

model.compile(loss = 'mse', optimizer = 'Adam', metrics = 'accuracy')

In [ ]:
model.fit(x_train, y_train_cat, batch_size = 64, epochs = 20, validation_split = 0.2)


In [ ]:
model.evaluate(x_test, y_test_cat)

In [ ]:

out = model.predict(x_test, batch_size=1)

In [ ]:
from skimage import transform
from skimage.io import imread

num_py = 32
num_px = 32

my_image = "fish.jpg"

# Загрузка указанного изображения
fname = "images/" + my_image
image1 = np.array(imread(fname))

# Предобработка и изменение размера изображения
image = image1/255.
image = transform.resize(image, [num_py,num_px,3])
my_image = image.reshape((1, num_py*num_px*3)).T
plt.imshow(image1)
print("Мнение нейронной сети: ", np.argmax(model.predict(my_image.reshape([1, 32, 32, 3]))))